# Day 5 — Model Explainability & Prototype Deployment

## Objectives
1. Generate **global** explainability (overall feature impact) and **patient-level** explainability (why a specific prediction was made) using **SHAP** and optionally **LIME**.
2. Validate the explanations for **clinical relevance** (sanity checks and consistency checks).
3. Package the selected model (XGBoost pipeline) into a **Gradio app** that outputs:
   - Predicted Final Stable Dose (mg)
   - A patient-level explanation (SHAP waterfall or top drivers)
   - Optional global summary plot

In [1]:
import sys
print(sys.executable)

C:\Users\caspe\anaconda\envs\genexa_ds\python.exe


In [2]:
# =========================
# Imports (Day 5)
# =========================
import os
import json
import random
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# ML / Metrics
from sklearn.model_selection import train_test_split

# Explainability
import shap
from lime.lime_tabular import LimeTabularExplainer

# MLflow (optional load/log)
import mlflow

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

C:\Users\caspe\anaconda\envs\genexa_ds\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pathlib import Path

# Adjust if your engineered dataset lives elsewhere
DATA_PATH = Path("../data/processed/merged_patient_dataset_engineered.csv")

if not DATA_PATH.exists():
    # fallback (your earlier structure showed curated folder)
    DATA_PATH = Path("../data/curated/merged_patient_dataset_engineered.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find engineered dataset at: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
TARGET_COL = "Final_Stable_Dose_mg"

# Basic checks
assert TARGET_COL in df.columns, f"Target column missing: {TARGET_COL}"
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
df.head()

Loaded: ..\data\processed\merged_patient_dataset_engineered.csv
Shape: (50000, 31)


,patient_id,CYP2C9,VKORC1,CYP4F2,Age,Sex,Weight_kg,Height_cm,Ethnicity,Hypertension,...,INR_Stabilization_Days,Adverse_Event,Time_in_Therapeutic_Range_Pct,Adverse_Event_Flag,CYP2C9_risk,VKORC1_sensitivity,CYP4F2_effect,BMI,Amiodarone_CYP2C9_interaction,Comorbidity_Score
0,P000001,*1/*3,A/G,C/C,64,F,88,194,Caucasian,0,...,5,NaN,66.8,0,1,1,0,23.381868,0,0
1,P000002,*1/*1,A/G,C/C,50,M,101,175,Other,1,...,8,NaN,72.4,0,0,1,0,32.979592,0,1
2,P000003,*1/*1,A/G,C/T,66,F,85,162,Asian,0,...,7,NaN,58.0,0,0,1,1,32.388355,0,0
3,P000004,*1/*2,G/G,C/T,58,M,83,178,African American,1,...,6,NaN,77.9,0,1,0,1,26.196187,0,1
4,P000005,*1/*1,G/G,C/T,61,F,75,194,Caucasian,1,...,7,NaN,70.6,0,0,0,1,19.927729,0,2


In [5]:
# Drop target from features
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (40000, 30) Test: (10000, 30)


In [9]:
# If you already trained it in this notebook/session:
# best_pipe.fit(X_train, y_train)
# (skip this cell if already fitted)

try:
    _ = best_pipe.predict(X_test.head(1))
    print("✅ best_pipe exists and is usable.")
except NameError:
    print("❌ best_pipe not found in memory. Use Option B (load from MLflow) below.")

❌ best_pipe not found in memory. Use Option B (load from MLflow) below.


In [13]:
# -------------------------
# MLflow model load option
# -------------------------
# Paste your best run_id here (from MLflow UI)
BEST_RUN_ID = "8f3eccd2cb8341a783fcaaf910e41c08"

# IMPORTANT: point tracking URI to the correct DB (yours is notebooks/mlflow.db)
mlflow.set_tracking_uri("sqlite:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlflow.db")

# This assumes you logged the model as an MLflow model artifact named "model"
# (you saw "model" under runs in the UI)
best_pipe = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")
print("Loaded best_pipe from MLflow model folder.")

UnpicklingError: invalid load key, 'v'.

In [7]:
from mlflow.tracking import MlflowClient
import mlflow
import os

BEST_RUN_ID = "8f3eccd2cb8341a783fcaaf910e41c08"  # yours

mlflow.set_tracking_uri(r"sqlite:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlflow.db")

client = MlflowClient()
run = client.get_run(BEST_RUN_ID)

print("artifact_uri:", run.info.artifact_uri)

# If artifact_uri is a local path like .../notebooks/mlruns/..., we can directly inspect the file

artifact_uri: file:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlruns/1/8f3eccd2cb8341a783fcaaf910e41c08/artifacts


In [8]:
# Try common locations inside the run artifacts
# If artifact_uri is like file:///C:/.../mlruns/<exp>/<run>/artifacts
artifact_path = run.info.artifact_uri.replace("file:///", "")
candidate = os.path.join(artifact_path, "model", "model.pkl")  # typical for mlflow.sklearn.log_model

print("Looking for:", candidate)
print("Exists:", os.path.exists(candidate))

if os.path.exists(candidate):
    with open(candidate, "rb") as f:
        head = f.read(120)
    print(head[:120])

Looking for: C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlruns/1/8f3eccd2cb8341a783fcaaf910e41c08/artifacts\model\model.pkl
Exists: False


In [10]:
from mlflow.tracking import MlflowClient
import mlflow

BEST_RUN_ID = "8f3eccd2cb8341a783fcaaf910e41c08"

mlflow.set_tracking_uri(r"sqlite:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlflow.db")
client = MlflowClient()

items = client.list_artifacts(BEST_RUN_ID)
print("Top-level artifacts:")
for it in items:
    print("-", it.path, "(dir)" if it.is_dir else "(file)")

Top-level artifacts:
- xgb_test_predictions.csv (file)


In [12]:
#Run this too:

# If you saw a folder named "model"
model_items = client.list_artifacts(BEST_RUN_ID, path="model")
print("\nInside /model:")
for it in model_items:
    print("-", it.path, "(dir)" if it.is_dir else "(file)")


Inside /model:
- model/MLmodel (file)
- model/conda.yaml (file)
- model/model.pkl (file)
- model/python_env.yaml (file)
- model/requirements.txt (file)


In [14]:
best_pipe = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")
print("Loaded best_pipe from MLflow model folder.")

UnpicklingError: invalid load key, 'v'.

In [17]:
import cloudpickle
import os

run = client.get_run(BEST_RUN_ID)
artifact_uri = run.info.artifact_uri  # file:///.../artifacts

artifact_path = artifact_uri.replace("file:///", "")
pkl_path = os.path.join(artifact_path, "model.pkl")  # change name to what you see from list_artifacts

with open(pkl_path, "rb") as f:
    best_pipe = cloudpickle.load(f)

print("✅ Loaded best_pipe from raw pickle artifact:", pkl_path)

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlruns/1/8f3eccd2cb8341a783fcaaf910e41c08/artifacts\\model.pkl'

In [18]:
from mlflow.tracking import MlflowClient
import mlflow, os

BEST_RUN_ID = "8f3eccd2cb8341a783fcaaf910e41c08"

mlflow.set_tracking_uri(r"sqlite:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlflow.db")
client = MlflowClient()

run = client.get_run(BEST_RUN_ID)
artifact_uri = run.info.artifact_uri  # file:///.../artifacts
artifact_path = artifact_uri.replace("file:///", "")

pkl_path = os.path.join(artifact_path, "model", "model.pkl")
print("PKL path:", pkl_path)
print("Exists:", os.path.exists(pkl_path))

with open(pkl_path, "rb") as f:
    head = f.read(120)

print(head)

PKL path: C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlruns/1/8f3eccd2cb8341a783fcaaf910e41c08/artifacts\model\model.pkl
Exists: False


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlruns/1/8f3eccd2cb8341a783fcaaf910e41c08/artifacts\\model\\model.pkl'

In [19]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(r"sqlite:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlflow.db")
client = MlflowClient()

# Your best run id
BEST_RUN_ID = "8f3eccd2cb8341a783fcaaf910e41c08"

# List registered models (if any)
models = client.search_registered_models()
print("Registered models:")
for m in models:
    print("-", m.name)

# List all model versions (this is what we need)
versions = client.search_model_versions("name like '%'")
print("\nModel versions:")
for v in versions:
    if v.run_id == BEST_RUN_ID:
        print("✅ Found model version linked to run:", v.name, "| version:", v.version, "| source:", v.source)

2025/12/14 18:45:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/14 18:45:23 INFO mlflow.store.db.utils: Updating database tables
2025/12/14 18:45:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/14 18:45:23 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Registered models:

Model versions:


In [20]:
import mlflow

mlflow.set_tracking_uri(r"sqlite:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlflow.db")

MODEL_DIR = r"C:\Users\caspe\OneDrive\Documents\genexahealth\genexahealth-data\notebooks\mlruns\models\m-4ca77d927acb4cea8a4610dc3e179a62\artifacts"

# Try loading as sklearn pipeline (best for SHAP because we can access preprocess + model)
try:
    best_pipe = mlflow.sklearn.load_model(MODEL_DIR)
    print("✅ Loaded as sklearn pipeline:", type(best_pipe))
except Exception as e:
    print("⚠️ sklearn load failed, trying pyfunc instead...")
    print("Error:", repr(e))
    best_pipe = mlflow.pyfunc.load_model(MODEL_DIR)
    print("✅ Loaded as MLflow pyfunc model:", type(best_pipe))

⚠️ sklearn load failed, trying pyfunc instead...
Error: MlflowException("Could not find a registered artifact repository for: c:. Currently registered schemes are: ['', 'file', 's3', 'r2', 'gs', 'wasbs', 'ftp', 'sftp', 'dbfs', 'hdfs', 'viewfs', 'runs', 'models', 'http', 'https', 'mlflow-artifacts', 'abfss']")


MlflowException: Could not find a registered artifact repository for: c:. Currently registered schemes are: ['', 'file', 's3', 'r2', 'gs', 'wasbs', 'ftp', 'sftp', 'dbfs', 'hdfs', 'viewfs', 'runs', 'models', 'http', 'https', 'mlflow-artifacts', 'abfss']

In [21]:
from pathlib import Path
import mlflow

mlflow.set_tracking_uri(r"sqlite:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlflow.db")

MODEL_DIR_WIN = r"C:\Users\caspe\OneDrive\Documents\genexahealth\genexahealth-data\notebooks\mlruns\models\m-4ca77d927acb4cea8a4610dc3e179a62\artifacts"
MODEL_DIR_URI = Path(MODEL_DIR_WIN).as_uri()

print("Model URI:", MODEL_DIR_URI)

Model URI: file:///C:/Users/caspe/OneDrive/Documents/genexahealth/genexahealth-data/notebooks/mlruns/models/m-4ca77d927acb4cea8a4610dc3e179a62/artifacts


In [22]:
# Try sklearn pipeline (best case)
try:
    best_pipe = mlflow.sklearn.load_model(MODEL_DIR_URI)
    print("✅ Loaded as sklearn pipeline:", type(best_pipe))
except Exception as e:
    print("⚠️ sklearn load failed, trying pyfunc...")
    print("Error:", repr(e))
    best_pipe = mlflow.pyfunc.load_model(MODEL_DIR_URI)
    print("✅ Loaded as pyfunc model:", type(best_pipe))

⚠️ sklearn load failed, trying pyfunc...
Error: OSError("No such file or directory: 'C:\\Users\\caspe\\OneDrive\\Documents\\genexahealth\\genexahealth-data\\notebooks\\mlruns\\models\\m-4ca77d927acb4cea8a4610dc3e179a62\\artifacts'")


OSError: No such file or directory: 'C:\Users\caspe\OneDrive\Documents\genexahealth\genexahealth-data\notebooks\mlruns\models\m-4ca77d927acb4cea8a4610dc3e179a62\artifacts'

In [23]:
preds = best_pipe.predict(X_test.head(3))
print("Predictions:", preds)

NameError: name 'best_pipe' is not defined